In [ ]:
import requests, geopandas as gpd

lon, lat = -93.65, 42.03  # your clicked point

# 1. point-in-polygon -> NHDPlus COMID
comid = requests.get(
    "https://api.water.usgs.gov/nldi/linked-data/comid/position",
    params={"coords": f"POINT({lon} {lat})", "f": "json"},
).json()["features"][0]["properties"]["comid"]

# 2. upstream drainage basin polygon
basin = gpd.read_file(
    f"https://api.water.usgs.gov/nldi/linked-data/comid/{comid}/basin"
)

In [ ]:
import basins

In [ ]:
lon, lat = -93.65, 42.03  # your clicked point
spatial_data = basins.delineate_basin(lat, lon)
basins.render_basin_map(spatial_data)

In [ ]:
import pandas as pd
sites_of_interest = "../data/IWQIS/sites_of_interest.csv"
usgs_sites_of_interest = "../data/USGS-NWIS/site_locations.csv"
def get_all_site_locations():
    iwqis = pd.read_csv(sites_of_interest, engine='python', on_bad_lines='warn')[["uid", "latitude", "longitude"]]
    usgs = pd.read_csv(usgs_sites_of_interest)
    print(usgs.columns)
    usgs.rename(columns={"monitoring_location_id" : "uid", "lat":"latitude", "lon":"longitude"}, inplace=True)
    print(usgs.columns)
    return pd.concat([iwqis, usgs])

locations_df = get_all_site_locations()
uids = list(locations_df.uid.unique())

locations_df = locations_df.set_index("uid")
pairs = {sid: (locations_df.loc[sid, "latitude"], locations_df.loc[sid, "longitude"]) for sid in locations_df.index.unique()}

pair = pairs[uids[0]]

thing = basins.delineate_basin(pair[0], pair[1])
print(type(thing))
print(thing.geometry)

In [ ]:
from pathlib import Path
geometry_path = Path("../data/USGS-NWIS/geometry/")
for uid in uids:
    lat, lon = pairs[uid][0], pairs[uid][1]
    thing = basins.delineate_basin(lat, lon)
    basins.save_basins(thing, path=geometry_path / f"{uid}_basins.parquet")